# Sensibilidad de los umbrales de accesibilidad en Sonora

Se usan 60 minutos para centros de salud y 120 para hospitales. Se compara la clasificación estricta `tiempo > umbral` al variar ambos cortes ±15 minutos. Un tiempo faltante queda fuera de la comparación. La población se cuenta por nivel y no se suma entre niveles.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

raiz = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/health_data").is_dir())
sys.path.insert(0, str(raiz / "src"))

from health_data.calidad_ruteo import tiempos_por_nivel, sensibilidad

procesados = Path(os.environ.get("HEALTH_DATA_PROCESSED_DIR", raiz / "data/processed"))
ruta = procesados / "accesibilidad_localidades.parquet"
tabla = pd.read_parquet(ruta)
tiempos = tiempos_por_nivel(tabla)
resumen, casos = sensibilidad(tiempos, margen_min=15)

medidas = ["localidades", "con_tiempo", "sobre_umbral", "cambia_umbral_menor",
           "cambia_umbral_mayor", "poblacion_cambia"]
display(resumen.groupby(["nivel", "metodo"])[medidas].sum())

## Municipios y localidades prioritarias

Se ordena el impacto municipal por población, y los casos individuales por población y cercanía al corte. Este orden facilita revisar las localidades con mayor población afectada.

In [ ]:
display(resumen.sort_values("poblacion_cambia", ascending=False).head(25))
display(casos[["locality_id", "locality_name", "municipality_name", "nivel", "metodo",
               "population", "tiempo_min", "distancia_al_umbral_min",
               "cambia_umbral_menor", "cambia_umbral_mayor"]].head(30))

## Tiempo disponible y respaldo CONEVAL

Las categorías CONEVAL describen el tiempo a un centro de servicios. Sirven para estratificar el análisis. Los minutos imputados a un establecimiento requieren rutas SAKBÉ de referencia. Si existen tiempos SAKBÉ o imputados, se comparan con la estimación por distancia en línea recta ajustada al mismo CLUES que figura en la tabla. El respaldo CONEVAL usa una mediana de grupo, de modo que el CLUES solo sirve de referencia para el escenario alternativo.

In [ ]:
display(tiempos.groupby(["nivel", "metodo", "tiempo_centro_servicios"],
                      dropna=False, observed=True)
        .agg(localidades=("nivel", "size"), poblacion=("population", "sum")))
if not tiempos["metodo"].eq("coneval").any():
    print("Este corte no contiene tiempos imputados con CONEVAL.")

from health_data.calidad_ruteo import comparar_con_estimacion

salud = procesados / "salud-001_establecimientos.parquet"
if salud.is_file() and tiempos["metodo"].isin(["sakbe", "coneval"]).any():
    pares = comparar_con_estimacion(tabla, pd.read_parquet(salud))
    display(pares.groupby(["nivel", "metodo", "municipality_name"], dropna=False)
            .agg(pares=("nivel", "size"), cambios=("cambia_clasificacion", "sum"),
                 poblacion_cambia=("poblacion_cambia", "sum")))
else:
    print("No hay datos locales suficientes para comparar escenarios emparejados.")